In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

dbutils.widgets.text("catalogo", "proyecto_ecommerce")
catalogo = dbutils.widgets.get("catalogo")

df_clientes = spark.table(f"{catalogo}.silver.clientes")
df_productos = spark.table(f"{catalogo}.silver.productos")
df_ordenes = spark.table(f"{catalogo}.silver.ordenes")

In [0]:
dim_region = (
    df_clientes.select("region").distinct()
    .withColumn("region_id", F.row_number().over(Window.orderBy("region")))
    .select("region_id", "region")
)

display(dim_region)
print(f"gold.dim_region (aún sin guardar) -> {dim_region.count()} filas")

In [0]:
(dim_region.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.gold.dim_region"))

print(f"Guardado: {catalogo}.gold.dim_region -> {dim_region.count()} filas")